<a href="https://colab.research.google.com/github/77MJohn77/Geospatial-Research-Portfolio/blob/main/Ruhango_Professional_DroughtMaps_2020_2025_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌍 Professional Drought Index Maps — Ruhango District (2020–2025)
### Monthly VCI · TCI · VHI Maps  |  May · June · July · August

**Study Area :** `projects/dotted-music-307621/assets/Ruhango`  
**GEE Project :** `dotted-music-307621`  
**Period :** 2020 – 2025  |  **Months :** May · June · July · August  
**Data :** MODIS MOD13Q1 (NDVI 250 m) · MOD11A2 (LST 1 km) · CHIRPS Rainfall  

---
**Each annual map contains 12 panels (3 indices × 4 months)**  
with North arrow · scale bar · drought-class legend — matching the Rusizi Watershed style.

| Class | Value |
|---|---|
| Extreme Drought | < 10 |
| Severe Drought | < 20 |
| Moderate Drought | < 30 |
| Mild Drought | < 40 |
| No Drought | ≥ 40 |

## ⚙️ STEP 1 — Install Packages & Authenticate GEE

In [ ]:
!pip install earthengine-api geemap requests pillow matplotlib numpy scipy -q
import ee, geemap

GEE_PROJECT = 'dotted-music-307621'
try:
    ee.Initialize(project=GEE_PROJECT)
    print(f'✅ GEE initialized — {GEE_PROJECT}')
except Exception:
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT)
    print(f'✅ GEE authenticated — {GEE_PROJECT}')

## 📦 STEP 2 — Imports & Configuration

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
import matplotlib.font_manager as fm
import requests
from PIL import Image
from io import BytesIO
import os, zipfile, warnings
warnings.filterwarnings('ignore')

# ── CONFIG ────────────────────────────────────────────────
ASSET         = 'projects/dotted-music-307621/assets/Ruhango'
GEE_PROJECT   = 'dotted-music-307621'
YEARS         = list(range(2020, 2026))        # 2020–2025
MONTHS        = [5, 6, 7, 8]                   # May Jun Jul Aug
MONTH_NAMES   = {5:'May', 6:'Jun', 7:'Jul', 8:'Aug'}
NDVI_SCALE    = 250
LST_SCALE     = 1000
VHI_ALPHA     = 0.5
EXPORT_FOLDER = 'Ruhango_Professional_Maps'
OUTPUT_DIR    = 'drought_maps_output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── DROUGHT COLOUR SCHEME  (matches reference image) ─────
DROUGHT_COLORS = {
    'Extreme':    '#8B0000',   # dark red
    'Severe':     '#FF4500',   # orange-red
    'Moderate':   '#FFD700',   # yellow
    'Mild':       '#90EE90',   # light green
    'No Drought': '#006400',   # dark green
}
CLASS_ORDER   = ['Extreme','Severe','Moderate','Mild','No Drought']
CMAP_VALUES   = [mcolors.to_rgb(DROUGHT_COLORS[k]) for k in CLASS_ORDER]
DROUGHT_CMAP  = mcolors.ListedColormap(CMAP_VALUES)
DROUGHT_NORM  = mcolors.BoundaryNorm([0.5,1.5,2.5,3.5,4.5,5.5], DROUGHT_CMAP.N)

# GEE palette string (no '#')
GEE_PALETTE   = [c.lstrip('#') for c in
                 [DROUGHT_COLORS[k] for k in CLASS_ORDER]]

print('✅ Configuration ready')
print(f'   Years  : {YEARS}')
print(f'   Months : {[MONTH_NAMES[m] for m in MONTHS]}')

## 🗺️ STEP 3 — Load Study Area

In [ ]:
AOI     = ee.FeatureCollection(ASSET)
AOI_GEO = AOI.geometry()
area_km2 = AOI_GEO.area().divide(1e6).getInfo()
print(f'✅ Loaded: Ruhango  |  Area = {area_km2:.1f} km²')

m = geemap.Map()
m.centerObject(AOI_GEO, 10)
m.addLayer(AOI_GEO, {'color':'0000FF','fillColor':'0000FF11'}, 'Ruhango')
m

## 📡 STEP 4 — Long-term Baseline (2001–2025) for NDVI & LST min/max

In [ ]:
def mask_ndvi(img):
    qa   = img.select('SummaryQA')
    ndvi = img.select('NDVI').multiply(0.0001)
    return ndvi.updateMask(qa.lte(1)).copyProperties(img, ['system:time_start'])

def mask_lst(img):
    qa  = img.select('QC_Day')
    lst = img.select('LST_Day_1km').multiply(0.02).subtract(273.15)
    return lst.updateMask(qa.bitwiseAnd(3).lte(1)).copyProperties(img, ['system:time_start'])

NDVI_BASE = (ee.ImageCollection('MODIS/006/MOD13Q1')
             .filterBounds(AOI_GEO).filterDate('2001-01-01','2025-12-31')
             .map(mask_ndvi))
LST_BASE  = (ee.ImageCollection('MODIS/006/MOD11A2')
             .filterBounds(AOI_GEO).filterDate('2001-01-01','2025-12-31')
             .select('LST_Day_1km').map(mask_lst))

NDVI_MIN = NDVI_BASE.min().clip(AOI_GEO)
NDVI_MAX = NDVI_BASE.max().clip(AOI_GEO)
LST_MIN  = LST_BASE.min().clip(AOI_GEO)
LST_MAX  = LST_BASE.max().clip(AOI_GEO)
print('✅ Baseline NDVI & LST min/max computed (2001–2025)')

## 🧮 STEP 5 — Index Computation Functions

In [ ]:
def get_ndvi_monthly(year, month):
    start = f'{year}-{month:02d}-01'
    end   = ee.Date(start).advance(1,'month').format('YYYY-MM-dd').getInfo()
    col   = (ee.ImageCollection('MODIS/006/MOD13Q1')
             .filterBounds(AOI_GEO).filterDate(start, end).map(mask_ndvi))
    return col.mean().clip(AOI_GEO)

def get_lst_monthly(year, month):
    start = f'{year}-{month:02d}-01'
    end   = ee.Date(start).advance(1,'month').format('YYYY-MM-dd').getInfo()
    col   = (ee.ImageCollection('MODIS/006/MOD11A2')
             .filterBounds(AOI_GEO).filterDate(start, end)
             .select('LST_Day_1km').map(mask_lst))
    return col.mean().clip(AOI_GEO)

def vci(ndvi):   return (ndvi.subtract(NDVI_MIN).divide(NDVI_MAX.subtract(NDVI_MIN))
                             .multiply(100).rename('VCI').clip(AOI_GEO))
def tci(lst):    return (LST_MAX.subtract(lst).divide(LST_MAX.subtract(LST_MIN))
                             .multiply(100).rename('TCI').clip(AOI_GEO))
def vhi(v, t):   return (v.multiply(VHI_ALPHA).add(t.multiply(1-VHI_ALPHA))
                             .rename('VHI').clip(AOI_GEO))

def classify(img, band):
    i = img.select(band)
    return (ee.Image(5).where(i.lt(40),4).where(i.lt(30),3)
                       .where(i.lt(20),2).where(i.lt(10),1)
                       .rename('cls').clip(AOI_GEO))

print('✅ Index functions ready')

## 🔄 STEP 6 — Compute All Monthly Indices (2020–2025)

In [ ]:
# results[year][month] = {VCI, TCI, VHI, VCI_cls, TCI_cls, VHI_cls}
results = {}

for year in YEARS:
    results[year] = {}
    print(f'── {year} ', end='')
    for month in MONTHS:
        ndvi_img = get_ndvi_monthly(year, month)
        lst_img  = get_lst_monthly(year, month)
        v = vci(ndvi_img)
        t = tci(lst_img)
        h = vhi(v, t)
        results[year][month] = {
            'VCI': v, 'TCI': t, 'VHI': h,
            'VCI_cls': classify(v,'VCI'),
            'TCI_cls': classify(t,'TCI'),
            'VHI_cls': classify(h,'VHI'),
        }
        print(f'{MONTH_NAMES[month]} ', end='', flush=True)
    print('✓')

print('\n✅ All indices computed!')

## 🖼️ STEP 7 — Helper: Fetch GEE Thumbnail

In [ ]:
VIS_CLS = {'min':1,'max':5,'palette':GEE_PALETTE}
VIS_IDX = {'min':0,'max':100,'palette':[
           '8B0000','FF4500','FF8C00','FFD700','ADFF2F','006400']}

def fetch_thumb(ee_img, vis, dims=400):
    """Download a GEE thumbnail and return a numpy RGBA array."""
    url = ee_img.getThumbURL({
        **vis,
        'region': AOI_GEO.bounds(),
        'dimensions': dims,
        'format': 'png'
    })
    resp = requests.get(url, timeout=120)
    resp.raise_for_status()
    return np.array(Image.open(BytesIO(resp.content)).convert('RGBA'))

print('✅ Thumbnail helper ready')

## 🗺️ STEP 8 — Professional Map Layout Functions
> Matches the **Rusizi Watershed** reference style:  
> — 3 columns (VCI · TCI · VHI) × 4 rows (May · Jun · Jul · Aug)  
> — North arrow · scale bar · drought-class legend · title banner

In [ ]:
def add_north_arrow(ax, x=0.93, y=0.94, size=0.12):
    """Draw a professional North arrow inside axes."""
    # Shaft
    ax.annotate('', xy=(x, y), xytext=(x, y - size),
                xycoords='axes fraction', textcoords='axes fraction',
                arrowprops=dict(arrowstyle='->', color='black', lw=1.8))
    # N label
    ax.text(x, y + 0.03, 'N', transform=ax.transAxes,
            ha='center', va='bottom', fontsize=9, fontweight='bold', color='black')

def add_scale_bar(ax, img_arr, aoi_geo, bar_km=25, pos='lower right'):
    """
    Draw a simple scale bar.  We estimate pixel-per-km from the AOI bounds.
    """
    try:
        coords = aoi_geo.bounds().getInfo()['coordinates'][0]
        lons = [c[0] for c in coords]; lats = [c[1] for c in coords]
        width_deg  = max(lons) - min(lons)
        lat_mid    = (max(lats) + min(lats)) / 2
        width_km   = width_deg * 111.32 * np.cos(np.radians(lat_mid))
        h, w       = img_arr.shape[:2]
        px_per_km  = w / width_km
        bar_px     = bar_km * px_per_km
        bar_frac   = bar_px / w
        # Positions
        x0 = 0.05; y0 = 0.04; bar_h = 0.015
        ax.add_patch(plt.Rectangle((x0, y0), bar_frac/2, bar_h,
                     transform=ax.transAxes, color='black', zorder=10))
        ax.add_patch(plt.Rectangle((x0 + bar_frac/2, y0), bar_frac/2, bar_h,
                     transform=ax.transAxes, color='white',
                     edgecolor='black', linewidth=0.8, zorder=10))
        ax.text(x0, y0 - 0.02, '0', transform=ax.transAxes,
                ha='center', va='top', fontsize=6.5, color='black')
        ax.text(x0 + bar_frac/2, y0 - 0.02, str(bar_km//2), transform=ax.transAxes,
                ha='center', va='top', fontsize=6.5, color='black')
        ax.text(x0 + bar_frac, y0 - 0.02, f'{bar_km} km', transform=ax.transAxes,
                ha='center', va='top', fontsize=6.5, color='black')
    except Exception:
        pass  # silently skip if bounds unavailable

def build_legend(fig, rect):
    """
    Draw a horizontal drought-class legend in the region `rect`
    = [left, bottom, width, height] in figure coords.
    Matches the reference image style.
    """
    ax_leg = fig.add_axes(rect)
    ax_leg.set_xlim(0, 1); ax_leg.set_ylim(0, 1)
    ax_leg.axis('off')

    # Title
    ax_leg.text(0.005, 0.78, 'Drought Class', fontsize=9, fontweight='bold', va='top')

    n = len(CLASS_ORDER)
    box_w = 0.10; gap = 0.005
    total = n * box_w + (n - 1) * gap
    x_start = 0.12
    for i, cls in enumerate(CLASS_ORDER):
        x = x_start + i * (box_w + gap)
        ax_leg.add_patch(plt.Rectangle((x, 0.15), box_w, 0.55,
                         transform=ax_leg.transAxes,
                         color=DROUGHT_COLORS[cls], clip_on=False,
                         edgecolor='grey', linewidth=0.5))
        ax_leg.text(x + box_w / 2, 0.0, cls,
                    transform=ax_leg.transAxes,
                    ha='center', va='top', fontsize=7.5)
    return ax_leg

print('✅ Map layout functions ready')

## 🗺️ STEP 9 — Generate & Save Professional Annual Maps
> **One publication-quality figure per year**  
> Layout: 4 rows (months) × 3 columns (VCI · TCI · VHI)

In [ ]:
saved_maps = []   # track saved filenames

for year in YEARS:
    print(f'\n═══════════════════════════')
    print(f'  Generating map for {year}')
    print(f'═══════════════════════════')

    # ── Layout: 4 rows × 3 cols + bottom legend strip ──
    nrows, ncols = 4, 3
    fig = plt.figure(figsize=(18, 22), facecolor='white')

    # Leave room at top for title and bottom for legend
    fig.subplots_adjust(top=0.93, bottom=0.07, left=0.03, right=0.97,
                        hspace=0.06, wspace=0.04)

    # ── Title banner ─────────────────────────────────────
    fig.text(0.5, 0.965,
             f'Drought Index Maps — Ruhango District — {year}',
             ha='center', va='top', fontsize=20, fontweight='bold',
             color='#1a1a1a',
             bbox=dict(boxstyle='round,pad=0.4', fc='#F0F0F0', ec='#333333', lw=1.2))

    # Column headers
    col_titles = ['VCI — Vegetation Condition Index',
                  'TCI — Temperature Condition Index',
                  'VHI — Vegetation Health Index']
    for ci, ct in enumerate(col_titles):
        fig.text(0.03 + (ci + 0.5) * (0.94/3), 0.945,
                 ct, ha='center', va='bottom',
                 fontsize=11, fontweight='bold', color='#333333')

    axes = []
    for ri, month in enumerate(MONTHS):
        row_axes = []
        for ci, index_name in enumerate(['VCI','TCI','VHI']):
            ax = fig.add_subplot(nrows, ncols, ri * ncols + ci + 1)

            cls_img = results[year][month][f'{index_name}_cls']
            print(f'  Fetching {index_name} {MONTH_NAMES[month]} {year}… ', end='', flush=True)

            try:
                thumb = fetch_thumb(cls_img, VIS_CLS, dims=512)
                # Replace white (background) with transparent for clean look
                white_mask = ((thumb[:,:,0]>240) & (thumb[:,:,1]>240) &
                              (thumb[:,:,2]>240))
                thumb[white_mask, 3] = 0
                ax.imshow(thumb, interpolation='bilinear')
                print('✓')
            except Exception as e:
                ax.set_facecolor('#CCCCCC')
                ax.text(0.5, 0.5, f'Data unavailable\n{str(e)[:40]}',
                        ha='center', va='center', transform=ax.transAxes,
                        fontsize=7, color='red')
                thumb = np.zeros((100,100,4), dtype=np.uint8)
                print(f'⚠ {e}')

            # ── Panel border & title ─────────────────────
            for sp in ax.spines.values():
                sp.set_edgecolor('#555555'); sp.set_linewidth(0.8)
            ax.set_xticks([]); ax.set_yticks([])

            # Row label (month) on left column only
            if ci == 0:
                ax.set_ylabel(f'{MONTH_NAMES[month]} {year}',
                              fontsize=10, fontweight='bold',
                              color='#222222', labelpad=4)

            # Panel subtitle
            ax.set_title(f'{index_name} — {MONTH_NAMES[month]} {year}',
                         fontsize=9, pad=3, color='#333333')

            # ── North arrow (top-right panel only) ───────
            if ri == 0 and ci == 2:
                add_north_arrow(ax, x=0.94, y=0.92, size=0.13)

            # ── Scale bar (bottom-right panel only) ──────
            if ri == 3 and ci == 2:
                add_scale_bar(ax, thumb, AOI_GEO, bar_km=25)

            row_axes.append(ax)
        axes.append(row_axes)

    # ── Legend strip at bottom ───────────────────────────
    build_legend(fig, [0.05, 0.012, 0.90, 0.055])

    # ── Data-source footnote ─────────────────────────────
    fig.text(0.5, 0.002,
             'Data: MODIS MOD13Q1 (NDVI 250m) · MOD11A2 (LST 1km) · '
             'CHIRPS Rainfall  |  Baseline: 2001–2025  |  '
             'GEE Project: dotted-music-307621',
             ha='center', va='bottom', fontsize=7, color='#666666', style='italic')

    # ── Save ─────────────────────────────────────────────
    fname = os.path.join(OUTPUT_DIR, f'Ruhango_DroughtMaps_{year}.png')
    fig.savefig(fname, dpi=200, bbox_inches='tight',
                facecolor='white', edgecolor='none')
    plt.show()
    plt.close(fig)
    saved_maps.append(fname)
    print(f'✅ Saved → {fname}')

print(f'\n🎉 All {len(saved_maps)} annual maps generated!')

## 🗺️ STEP 10 — Per-Index Summary Maps (VCI / TCI / VHI across all years)

In [ ]:
for index_name in ['VCI','TCI','VHI']:
    print(f'\nGenerating {index_name} summary map (all years × 4 months)…')

    nrows = len(YEARS)     # 6
    ncols = len(MONTHS)    # 4

    fig = plt.figure(figsize=(18, 26), facecolor='white')
    fig.subplots_adjust(top=0.93, bottom=0.07, left=0.04, right=0.97,
                        hspace=0.05, wspace=0.04)

    fig.text(0.5, 0.965,
             f'{index_name} Drought Index Maps — Ruhango District (2020–2025)',
             ha='center', va='top', fontsize=18, fontweight='bold',
             bbox=dict(boxstyle='round,pad=0.4', fc='#F0F0F0', ec='#333333', lw=1.2))

    # Column headers
    for ci, month in enumerate(MONTHS):
        fig.text(0.04 + (ci + 0.5) * (0.93/4), 0.945,
                 MONTH_NAMES[month], ha='center', va='bottom',
                 fontsize=12, fontweight='bold', color='#333333')

    for ri, year in enumerate(YEARS):
        for ci, month in enumerate(MONTHS):
            ax = fig.add_subplot(nrows, ncols, ri * ncols + ci + 1)
            cls_img = results[year][month][f'{index_name}_cls']

            try:
                thumb = fetch_thumb(cls_img, VIS_CLS, dims=400)
                white = (thumb[:,:,0]>240)&(thumb[:,:,1]>240)&(thumb[:,:,2]>240)
                thumb[white, 3] = 0
                ax.imshow(thumb, interpolation='bilinear')
            except Exception as e:
                ax.set_facecolor('#DDDDDD')
                ax.text(0.5,0.5,'N/A',ha='center',va='center',
                        transform=ax.transAxes,fontsize=8,color='red')

            for sp in ax.spines.values():
                sp.set_edgecolor('#555555'); sp.set_linewidth(0.7)
            ax.set_xticks([]); ax.set_yticks([])
            ax.set_title(f'{MONTH_NAMES[month]} {year}', fontsize=8, pad=2)

            if ci == 0:
                ax.set_ylabel(str(year), fontsize=10, fontweight='bold', labelpad=4)

            # North arrow top-right
            if ri == 0 and ci == ncols - 1:
                add_north_arrow(ax, x=0.93, y=0.91)
            # Scale bar bottom-right
            if ri == nrows - 1 and ci == ncols - 1:
                add_scale_bar(ax, np.zeros((400,400,4)), AOI_GEO, bar_km=25)

    build_legend(fig, [0.05, 0.012, 0.90, 0.05])
    fig.text(0.5, 0.002,
             'Source: MODIS/Google Earth Engine  |  GEE: dotted-music-307621',
             ha='center', fontsize=7, color='#666666', style='italic')

    fname = os.path.join(OUTPUT_DIR, f'Ruhango_{index_name}_AllYears_2020_2025.png')
    fig.savefig(fname, dpi=180, bbox_inches='tight', facecolor='white')
    plt.show(); plt.close(fig)
    saved_maps.append(fname)
    print(f'✅ Saved → {fname}')

## 📤 STEP 11 — Export GeoTIFFs to Google Drive

In [ ]:
def export_tif(image, name, folder, scale, region):
    task = ee.batch.Export.image.toDrive(
        image=image, description=name, folder=folder,
        fileNamePrefix=name, scale=scale,
        region=region, crs='EPSG:4326',
        maxPixels=1e13, fileFormat='GeoTIFF'
    )
    task.start()
    return task

all_tasks = []
print(f'Submitting GeoTIFF export tasks → Drive/{EXPORT_FOLDER}\n')

for year in YEARS:
    for month in MONTHS:
        mn = MONTH_NAMES[month]
        r  = results[year][month]
        for idx_name in ['VCI','TCI','VHI']:
            # Continuous index
            t = export_tif(r[idx_name].toFloat(),
                           f'{idx_name}_{year}_{mn}_Ruhango',
                           EXPORT_FOLDER, NDVI_SCALE, AOI_GEO)
            all_tasks.append(t)
            # Classification map
            t2 = export_tif(r[f'{idx_name}_cls'].toUint8(),
                            f'{idx_name}_Class_{year}_{mn}_Ruhango',
                            EXPORT_FOLDER, NDVI_SCALE, AOI_GEO)
            all_tasks.append(t2)

print(f'✅ {len(all_tasks)} export tasks submitted!')
print(f'Monitor: https://code.earthengine.google.com/tasks')
print(f'Files → Google Drive / {EXPORT_FOLDER}/')

## ⏳ STEP 12 — Monitor Export Tasks

In [ ]:
import time

def monitor_tasks(tasks, interval=30, timeout_min=90):
    deadline = time.time() + timeout_min * 60
    while time.time() < deadline:
        states = [t.status()['state'] for t in tasks]
        done    = states.count('COMPLETED')
        failed  = states.count('FAILED')
        running = sum(1 for s in states if s in ('RUNNING','READY','SUBMITTED'))
        print(f'[{time.strftime("%H:%M:%S")}]  '
              f'Done={done}/{len(tasks)}  Running={running}  Failed={failed}')
        if running == 0:
            print('✅ All tasks finished!'); break
        time.sleep(interval)
    else:
        print(f'⏰ Timeout — check manually: https://code.earthengine.google.com/tasks')

# ── Uncomment to actively monitor: ──
# monitor_tasks(all_tasks, interval=30, timeout_min=90)
print('Uncomment monitor_tasks() above to poll task status.')

## 💾 STEP 13 — Download All PNG Maps as ZIP

In [ ]:
from google.colab import files as colab_files

zip_path = 'Ruhango_Professional_DroughtMaps_2020_2025.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fpath in saved_maps:
        if os.path.exists(fpath):
            zf.write(fpath, os.path.basename(fpath))
            print(f'  ✔ {os.path.basename(fpath)}')
        else:
            print(f'  ⚠ Not found: {fpath}')

print(f'\n✅ ZIP created: {zip_path}')
colab_files.download(zip_path)
print('⬇️  Download started!')

## 📥 STEP 14 — Download GeoTIFFs from Google Drive

In [ ]:
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

drive_path = f'/content/drive/MyDrive/{EXPORT_FOLDER}'

if os.path.exists(drive_path):
    tifs = sorted([f for f in os.listdir(drive_path) if f.endswith('.tif')])
    print(f'Found {len(tifs)} GeoTIFF files:\n')
    total_mb = 0
    for tf in tifs:
        mb = os.path.getsize(os.path.join(drive_path, tf)) / 1e6
        total_mb += mb
        print(f'  {tf:65s} {mb:.2f} MB')
    print(f'\n  Total: {total_mb:.1f} MB')

    tif_zip = 'Ruhango_GeoTIFF_Maps_2020_2025.zip'
    with zipfile.ZipFile(tif_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
        for tf in tifs:
            zf.write(os.path.join(drive_path, tf), tf)
    print(f'\n✅ ZIP: {tif_zip}')
    colab_files.download(tif_zip)
    print('⬇️  GeoTIFF download started!')
else:
    print(f'⚠️  Folder not found: {drive_path}')
    print('    Run Step 12 (export) and wait for tasks to complete first.')

---
## 📚 References
- Kogan (1995a,b) — VCI & TCI formulations
- Bhuiyan et al. (2017) — VHI formula
- Burka et al. (2024) — Spatial drought mapping methodology using GEE
- Gorelick et al. (2017) — Google Earth Engine platform

---
*Study Area: Ruhango District, Rwanda | GEE Project: `dotted-music-307621`*